# Welcome to Colab!

# setup

In [ ]:
!python --version


Python 3.12.12


In [ ]:
!git fetch

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
!git pull origin HongPhuc

fatal: not a git repository (or any of the parent directories): .git




```
# Định dạng của đoạn này là mã
```



# Code

# CREATE DATASET FOR CROSS-ENCODE

In [ ]:
# CELL 1 - Install
!python -m pip -q install -U "transformers>=4.41.0" "accelerate>=0.30.0" "bitsandbytes>=0.46.1" "sentencepiece"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00


In [ ]:
# CELL 2 - Quick check GPU + versions
import torch, sys
print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CUDA available: True
GPU: Tesla T4
Torch: 2.10.0+cu128 CUDA: 12.8


In [ ]:
# CELL 3 — Mount Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 4 — Load chunks
import json

CHUNKS_PATH = "/content/drive/MyDrive/Legal chat bot/Data/chunks.json"
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print("Total chunks:", len(chunks))
print("Chunk keys:", chunks[0].keys())
print("Metadata keys:", chunks[0].get("metadata", {}).keys())

Total chunks: 1861
Chunk keys: dict_keys(['text', 'metadata'])
Metadata keys: dict_keys(['van_ban', 'chuong', 'dieu', 'khoan', 'diem', 'source_file'])


In [ ]:
# CELL 5 — Load model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

# giảm VRAM khi generate
model.config.use_cache = False

print("Model loaded. Device:", next(model.parameters()).device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded. Device: cuda:0


In [ ]:
# CELL 6 — build_prompt (plain text) + system message
import json

# ⚠️ Qwen là model đa ngôn ngữ, đôi khi "trôi" sang tiếng Trung nếu ràng buộc chưa đủ chặt.
# SYSTEM_MSG dưới đây siết chặt yêu cầu: CHỈ tiếng Việt, cấm ký tự Hán (CJK Han).
SYSTEM_MSG = (
    "Bạn là công cụ sinh câu hỏi CHỈ BẰNG TIẾNG VIỆT dựa trên đoạn trích pháp luật Việt Nam. "
    "BẮT BUỘC: chỉ dùng tiếng Việt (có dấu), TUYỆT ĐỐI KHÔNG dùng tiếng Trung/English hay ký tự Hán. "
    "Nếu bạn lỡ viết ra bất kỳ ký tự Hán (CJK Han) hoặc ngôn ngữ khác, hãy tự sửa và xuất lại bằng tiếng Việt trước khi trả lời. "
    "Không giải thích. Không markdown. Không thêm text ngoài JSON."
)

def build_prompt(chunk, n_questions=4):
    md = chunk.get("metadata", {}) or {}
    doc = md.get("van_ban", "Văn bản pháp luật")
    chuong = md.get("chuong")
    dieu = md.get("dieu")
    khoan = md.get("khoan")
    diem = md.get("diem")

    ref_parts = []
    if chuong: ref_parts.append(f"Chương {chuong}")
    if dieu:   ref_parts.append(f"Điều {dieu}")
    if khoan:  ref_parts.append(f"Khoản {khoan}")
    if diem:   ref_parts.append(f"Điểm {diem}")
    ref = " - ".join(ref_parts) if ref_parts else "Không rõ tham chiếu"

    passage = chunk.get("text", "")

    return f"""
Văn bản: {doc}
Tham chiếu: {ref}

Đoạn trích:
\"\"\"{passage}\"\"\"

Yêu cầu:
- Sinh {n_questions} câu hỏi tiếng Việt tự nhiên mà người dùng có thể hỏi.
- Câu hỏi phải trả lời được chỉ dựa trên đoạn trích.
- Không nhắc "đoạn trích", "đoạn văn này", "dựa vào đoạn".
- Không thêm giải thích, không thêm markdown.
- TUYỆT ĐỐI KHÔNG dùng tiếng Trung/English hoặc ký tự Hán.

Trả về DUY NHẤT JSON đúng schema:
{{"queries": ["...", "..."]}}
""".strip()


In [ ]:
# CELL 7 — safe_parse_queries (robust)
import json

def safe_parse_queries(output_text):
    # thử parse trực tiếp nếu output là JSON sạch
    s = output_text.strip()
    try:
        data = json.loads(s)
        qs = data.get("queries", [])
        if isinstance(qs, list):
            return [q.strip() for q in qs if isinstance(q, str) and len(q.strip()) >= 10]
    except Exception:
        pass

    # fallback: tìm JSON object bằng cách quét từ cuối lên
    last_r = output_text.rfind("}")
    if last_r == -1:
        return []
    for start in range(output_text.rfind("{", 0, last_r), -1, -1):
        if output_text[start] != "{":
            continue
        blob = output_text[start:last_r+1]
        try:
            data = json.loads(blob)
            qs = data.get("queries", [])
            if isinstance(qs, list):
                return [q.strip() for q in qs if isinstance(q, str) and len(q.strip()) >= 10]
        except Exception:
            continue
    return []


In [ ]:
# CELL 8 — Lọc query rác (nhẹ, không chặn quá mạnh) + chặn ký tự Hán
import re

BAD_PATTERNS = [
    r"\bbạn là\b", r"\bdựa vào\b", r"\bđoạn trích\b", r"\bđoạn văn\b",
    r"\bhãy tạo\b", r"\bsinh câu hỏi\b", r"\btheo đoạn\b"
]

# ký tự Hán (CJK Unified Ideographs) — đủ để bắt các trường hợp tiếng Trung hay gặp
HAN_RE = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]")

def is_valid_query(q: str) -> bool:
    q = q.strip()
    if len(q) < 10:
        return False

    # ❌ loại nếu có ký tự Hán (tiếng Trung/Japanese Kanji…)
    if HAN_RE.search(q):
        return False

    if not q.endswith("?"):
        # vẫn chấp nhận nhưng ưu tiên có dấu hỏi
        pass

    low = q.lower()
    for pat in BAD_PATTERNS:
        if re.search(pat, low):
            return False
    return True


In [ ]:
import torch
import re

# ký tự Hán (CJK) để phát hiện output "trôi" sang tiếng Trung
HAN_RE = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF]")

def _contains_han(text: str) -> bool:
    return bool(HAN_RE.search(text or ""))

@torch.no_grad()
def generate_queries(chunk, n_questions=4, max_new_tokens=256, temperature=0.2, max_retries=2):
    """Generate câu hỏi (JSON) và tự retry nếu phát hiện ký tự Hán."""
    user_prompt = build_prompt(chunk, n_questions=n_questions)

    def _gen_once(sys_msg: str, temp: float):
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": user_prompt},
        ]

        enc = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        )

        # ✅ handle cả 2 kiểu: Tensor hoặc BatchEncoding
        if isinstance(enc, torch.Tensor):
            input_ids = enc.to(model.device)
            attention_mask = None
            prompt_len = input_ids.shape[-1]
        else:
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc.get("attention_mask")
            if attention_mask is not None:
                attention_mask = attention_mask.to(model.device)
            prompt_len = input_ids.shape[-1]

        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temp,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_ids = out[0][prompt_len:]  # ✅ cắt đúng phần mới sinh
        return tokenizer.decode(gen_ids, skip_special_tokens=True)

    # thử 1 lần bình thường
    out_text = _gen_once(SYSTEM_MSG, temperature)

    # nếu có ký tự Hán → retry với system message "cứng" hơn + nhiệt thấp hơn
    tries = 0
    while tries < max_retries and _contains_han(out_text):
        tries += 1
        strict_msg = (
            SYSTEM_MSG
            + " VI PHẠM NGÔN NGỮ Ở LẦN TRƯỚC. "
            + "Hãy xuất lại CHỈ tiếng Việt, KHÔNG chứa bất kỳ ký tự Hán nào. "
            + "Trả về DUY NHẤT JSON theo schema."
        )
        out_text = _gen_once(strict_msg, temp=max(0.05, temperature - 0.1 * tries))

    return out_text


In [ ]:
# CELL 10 — Clean GPU helper
import gc, torch

def clean_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# CELL 11 — Tạo dataset (JSONL) + RESUME
import os, json, time

OUT_DIR = "/content/drive/MyDrive/Legal chat bot/Data"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(OUT_DIR, "train.jsonl")
DEV_PATH   = os.path.join(OUT_DIR, "dev.jsonl")

# cấu hình
NQ = 4
MAX_NEW = 256
SAVE_EVERY = 25
DEV_RATIO = 0.1

def write_jsonl(path, rows, mode="a"):
    with open(path, mode, encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def count_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

def get_last_chunk_index(path):
    """
    Đọc ngược cuối file jsonl để lấy meta.chunk_index lớn nhất.
    Tránh đọc toàn bộ file nếu file lớn.
    """
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return -1

    # đọc vài KB cuối file
    with open(path, "rb") as f:
        f.seek(0, os.SEEK_END)
        size = f.tell()
        step = 8192
        data = b""
        pos = max(0, size - step)
        f.seek(pos)
        data = f.read(size - pos)

    lines = data.splitlines()[::-1]  # đảo ngược
    for line in lines:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line.decode("utf-8"))
            idx = (obj.get("meta") or {}).get("chunk_index")
            if isinstance(idx, int):
                return idx
        except:
            continue
    return -1

# ✅ RESUME: xác định chunk đã chạy tới đâu
last_train_idx = get_last_chunk_index(TRAIN_PATH)
last_dev_idx   = get_last_chunk_index(DEV_PATH)
last_done_idx  = max(last_train_idx, last_dev_idx)  # chunk_index (0-based)

start_i = last_done_idx + 2  # convert sang i (1-based) cho enumerate(start=1)
if start_i < 1:
    start_i = 1

print("Resume from chunk i =", start_i, "(1-based)")
print("Train lines:", count_lines(TRAIN_PATH), "Dev lines:", count_lines(DEV_PATH))

total = len(chunks)
t0 = time.time()

for i in range(start_i, total + 1):
    chunk = chunks[i - 1]  # i is 1-based
    print(f"[{i}/{total}] generating...")

    output_text = generate_queries(chunk, n_questions=NQ, max_new_tokens=MAX_NEW)
    queries = safe_parse_queries(output_text)

    if not queries:
        clean_gpu()
        continue

    md = chunk.get("metadata", {}) or {}

    rows = []
    for q in queries:
        if not is_valid_query(q):
            continue
        rows.append({
            "query": q,
            "passage": chunk.get("text",""),
            "label": 1,
            "meta": {
                "van_ban": md.get("van_ban"),
                "chuong": md.get("chuong"),
                "dieu": md.get("dieu"),
                "khoan": md.get("khoan"),
                "diem": md.get("diem"),
                "source_file": md.get("source_file"),
                "chunk_index": i-1
            }
        })

    if rows:
        import random
        for r in rows:
            if random.random() < DEV_RATIO:
                write_jsonl(DEV_PATH, [r], mode="a")
            else:
                write_jsonl(TRAIN_PATH, [r], mode="a")

    clean_gpu()

    if i % SAVE_EVERY == 0:
        elapsed = time.time() - t0
        print(f"Checkpoint at chunk {i}. Elapsed: {elapsed:.1f}s")

print("DONE.")
print("Train:", TRAIN_PATH)
print("Dev:", DEV_PATH)

Resume from chunk i = 1134 (1-based)
Train lines: 4013 Dev lines: 447
[1134/1861] generating...
[1135/1861] generating...
[1136/1861] generating...
[1137/1861] generating...
[1138/1861] generating...
[1139/1861] generating...
[1140/1861] generating...
[1141/1861] generating...
[1142/1861] generating...
[1143/1861] generating...
[1144/1861] generating...
[1145/1861] generating...
[1146/1861] generating...
[1147/1861] generating...
[1148/1861] generating...
[1149/1861] generating...
[1150/1861] generating...
Checkpoint at chunk 1150. Elapsed: 180.6s
[1151/1861] generating...
[1152/1861] generating...
[1153/1861] generating...
[1154/1861] generating...
[1155/1861] generating...
[1156/1861] generating...
[1157/1861] generating...
[1158/1861] generating...
[1159/1861] generating...
[1160/1861] generating...
[1161/1861] generating...
[1162/1861] generating...
[1163/1861] generating...
[1164/1861] generating...
[1165/1861] generating...
[1166/1861] generating...
[1167/1861] generating...
[116

In [ ]:
# CELL 12 (optional) — add negatives (same van_ban preferred)
import json, random, os

NEG_TRAIN_PATH = os.path.join(OUT_DIR, "train_with_neg.jsonl")

# index chunks theo van_ban để lấy hard negatives
by_doc = {}
for idx, ch in enumerate(chunks):
    doc = (ch.get("metadata") or {}).get("van_ban", "UNKNOWN")
    by_doc.setdefault(doc, []).append(idx)

def pick_negative(chunk_idx):
    ch = chunks[chunk_idx]
    doc = (ch.get("metadata") or {}).get("van_ban", "UNKNOWN")
    candidates = by_doc.get(doc, [])
    if len(candidates) >= 2:
        # chọn chunk khác cùng văn bản
        j = random.choice([x for x in candidates if x != chunk_idx])
    else:
        # fallback random toàn cục
        j = random.randrange(len(chunks))
        while j == chunk_idx:
            j = random.randrange(len(chunks))
    return chunks[j].get("text","")

# đọc train.jsonl và tạo thêm 1 negative cho mỗi positive
open(NEG_TRAIN_PATH, "w", encoding="utf-8").close()

with open(TRAIN_PATH, "r", encoding="utf-8") as f_in, open(NEG_TRAIN_PATH, "a", encoding="utf-8") as f_out:
    for line in f_in:
        pos = json.loads(line)
        chunk_idx = (pos.get("meta") or {}).get("chunk_index")
        if chunk_idx is None:
            continue

        # ghi lại positive
        f_out.write(json.dumps(pos, ensure_ascii=False) + "\n")

        # tạo negative
        neg = dict(pos)
        neg["passage"] = pick_negative(chunk_idx)
        neg["label"] = 0
        f_out.write(json.dumps(neg, ensure_ascii=False) + "\n")

print("Wrote:", NEG_TRAIN_PATH)


Wrote: /content/drive/MyDrive/Legal chat bot/Data/train_with_neg.jsonl
